In [5]:
from urllib.request import Request, urlopen 
from bs4 import BeautifulSoup

def remove_content_between_chars(string: str, start="[", end="]"):
    edited = []
    skip = False
    for char in string:
        if char == start:
            skip = True

        if not skip:
            edited.append(char)

        if char == end:
            assert skip, f"Closing character found without opening in {''.join(edited)}"
            skip = False
    return "".join(edited)


In [6]:
import csv
import json
import re
from typing import List, Set

from spacy.tokens.doc import Doc
from spacy.tokens import Span
from tqdm import tqdm
from unidecode import unidecode

def get_paragraphs(url: str):
        headers = {
        "User-Agent": "Mozilla/5.0 (compatible; WikipediaScraper/1.0; +https://example.com/contact)"
        }
        req = Request(url, headers=headers)
        page = urlopen(req)

        assert page.getcode() == 200, f"{url} is not valid"
        soup = BeautifulSoup(page, "html.parser")

        res = [soup.find("h1")]
        div = soup.find("div", class_="mw-content-ltr mw-parser-output")
        res += [i for i in div.find_all(re.compile("(h.)|p"))]

        paragraphs = []
        for tag in res:
            if tag.name.startswith("h"):
                current_header = tag
            elif tag.name == "p" and tag.get("style") is None:
                # remove multiple whitespaces
                p = " ".join(tag.text.split())
                # remove additional whitespaces
                p = p.strip()
                # remove content between '[' and ']'
                p = remove_content_between_chars(p)
                if len(p) > 0:
                    paragraphs.append({
                        "text": p,
                        "header_name": current_header.text,
                        "header_level": current_header.name[-1]
                    })
        
        return paragraphs
        
        
def check_exact_match(paragraph: str, answer: str):
    # case insensitive match
    return bool(re.search(f"(^|[^\w]{{1}}){answer}($|[^\w]{{1}})", paragraph, flags=re.IGNORECASE))


def check_full_match(paragraph: str, attribute: str, answer: str):
    # a full match requires to match both attribute (e.g. President) and answers
    for to_find in [attribute, answer]:
        if not check_exact_match(paragraph, to_find):
            return False
    return True


def write_roman(num: int):
    ROMAN = {
        1000: "M",
        900: "CM",
        500: "D",
        400: "CD",
        100: "C",
        90: "XC",
        50: "L",
        40: "XL",
        10: "X",
        9: "IX",
        5: "V",
        4: "IV",
        1: "I",
    }

    def roman_num(num: int):
        for r in ROMAN.keys():
            x, y = divmod(num, r)
            yield ROMAN[r] * x
            num -= r * x
            if num <= 0:
                break

    return "".join([a for a in roman_num(num)])


def remove_additional_bits(string: str, additional_bits: List[str]):
    for bit in additional_bits:
        string = re.sub(bit, "", string)
    return " ".join(string.split())  # remove additional whitespaces


def find_main_chunk(doc: Doc):
    ancestor = None
    for chunk in doc.noun_chunks:
        if ancestor is None:
            ancestor = chunk
        elif chunk.root.is_ancestor(ancestor.root):
            ancestor = chunk.root
    return ancestor


def is_monarch(span: Span, monarch_nums: Set[str]):
    for name_chunk in span.text.split():
        if name_chunk in monarch_nums:
            return True
    return False

In [7]:
import spacy
from spacy.tokenizer import Tokenizer

MONARCH_NUMS = {write_roman(i) for i in range(1, 100, 1)}

ADDITIONAL_BITS = [
    "[\w]?F(\.)?C(\.)?[\w]?",  # FC, F.C., AFC, ... for Football
    "[\w]?C(\.)?F(\.)?[\w]?",  # CF, ... for Football
    "[\w]?F(\.)?K(\.)?[\w]?",  # FK, ... for Football
    "[\w]?A(\.)?S(\.)?[\w]?",  # AS, ... for Football
    "[\w]?S(\.)?V(\.)?[\w]?",  # SV, ... for Football
    "[\w]?B(\.)?C(\.)?[\w]?",  # BC, ... for Basketball
    "[\w](\.)[\w](\.)",  # General regex for to remove two letter acronyms (with )
    "football",
    "(t|T)eam",
    "association",
    "men's",
    "basketball",
    "F1",
    "(S|s)cuderia",
    "(R|r)acing",
]

EXCEPTIONS_ANSWERS = {
    "Kōji Seto": "Koji Sato", 
    "Salman bin Abdulaziz Al Saud": "Salman" #The head noun is selected as Saud wrt Salman; Salman in wikidata is both king and prime minister while in wikipedia no
    } 

nlp = spacy.load("en_core_web_trf")
nlp.tokenizer = Tokenizer(nlp.vocab)  # Whitespace tokenization

passages = {}
with open("wikipedia_pages.csv", "r") as f:
    lines = f.readlines()

with open("wikipedia_pages.csv", "r") as f:
    reader = csv.DictReader(f)

    for row in tqdm(reader, total=len(lines)-1):
        category = row["category"]
        item = row["item"]
        page_url = row["page_url"]
        attribute = row["attribute"]
        attribute = attribute if attribute != "Sports Team" else None
        
        if category not in passages:
            passages[category] = {}
        
        if item not in passages[category]:
            passages[category][item] = {}

        if attribute is not None:
            if attribute not in passages[category][item]:
                passages[category][item][attribute] = {}

        # remove unicode characters from url
        url = unidecode(page_url)
        paragraphs = get_paragraphs(url)

        matches = {
            "full": [],
            "em": [],
            "simplified": [],
            "head": [],
        }

        no_match = True
        for paragraph in tqdm(paragraphs, desc=f"paragraphs for {item}"):
            answer = row["answer"]
            if answer in EXCEPTIONS_ANSWERS:
                answer = EXCEPTIONS_ANSWERS[answer]
            append_to = None
            matched = None

            attr = attribute
            if attribute is not None:
                if "prime minister" in attr.lower():
                    attr = "prime minister"
                if "president" in attr.lower():
                    attr = "president"
                if "chancellor" in attr.lower():
                    attr = "chancellor"
                if "king" in attr.lower():
                    attr = "king"
                if "emperor" in attr.lower():
                    attr = "emperor"
                if "monarch" in attr.lower():
                    attr = "monarch"
                if "supreme leader" in attr.lower():
                    attr = "supreme leader"
                if "premier" in attr.lower():
                    attr = "premier"
            if check_full_match(paragraph["text"], attr, answer):
                append_to = "full"
                matched = (attr, answer)
                no_match = False
            elif check_exact_match(paragraph["text"], answer):
                append_to = "em"
                matched = answer
                no_match = False
            else:
                if category in ["athletes"]:
                    answer = remove_additional_bits(answer, ADDITIONAL_BITS)

                if check_exact_match(paragraph["text"], answer):
                    append_to = "simplified"
                    matched = answer
                    no_match = False
                elif len(answer.split()) > 1:
                    doc = nlp(answer)
                    main_chunk = find_main_chunk(doc)

                    if main_chunk is not None:
                        if is_monarch(main_chunk, MONARCH_NUMS):
                            answer = main_chunk.text
                        else:
                            answer = main_chunk.root.text

                    if check_exact_match(paragraph["text"], answer):
                        append_to = "head"
                        #print(answer)
                        matched = answer
                        no_match = False

            
            if append_to:
                matches[append_to].append({
                    "paragraph": paragraph,
                    "matched": matched
                })

        if attribute is not None:
            if attribute not in passages[category][item]:
                passages[category][item][attribute] = {}

            passages[category][item][attribute] = {
                "matches": matches,
                "page_url": url,
                "no_match": no_match
            }
        else:
            passages[category][item] = {
                "matches": matches,
                "page_url": url,
                "no_match": no_match
            }



with open("passages.json", "w") as f:
    json.dump(passages, f, indent=4)


100%|██████████| 55/55 [02:19<00:00,  2.54s/it]


In [8]:
with open("passages.json", "r") as f:
    passages = json.load(f)

refined_passages = {}

for domain in passages:
    for element in passages[domain]:

        if domain == "athletes":
            data = passages[domain][element]

            if "matches" not in data:
                raise ValueError(f"No matches found for {element} in domain {domain}")

            matches = data["matches"]
            index = "|".join([domain, element])

            match_type = None
            for mt in ["full", "em", "simplified", "head"]:
                if mt in matches and len(matches[mt]) > 0:
                    match_type = mt
                    break

            if match_type is None:
                raise ValueError(f"No matches found for {element} in domain {domain}")

            refined_passages[index] = {
                "text": matches[match_type][0]["paragraph"]["text"],
                "matched": matches[match_type][0]["matched"],
                "match_type": match_type,
                "page_url": data["page_url"],
            }

        else:
            for attribute in passages[domain][element]:
                data = passages[domain][element][attribute]

                if "matches" not in data:
                    raise ValueError(
                        f"No matches found for {element} - {attribute} in domain {domain}"
                    )

                matches = data["matches"]
                index = "|".join([domain, element, attribute])

                match_type = None
                for mt in ["full", "em", "simplified", "head"]:
                    if mt in matches and len(matches[mt]) > 0:
                        match_type = mt
                        break

                if match_type is None:
                    raise ValueError(
                        f"No matches found for {element} - {attribute} in domain {domain}"
                    )

                refined_passages[index] = {
                    "text": matches[match_type][0]["paragraph"]["text"],
                    "matched": matches[match_type][0]["matched"],
                    "match_type": match_type,
                    "page_url": data["page_url"],
                }
                

texts = set()
keys = list(refined_passages.keys())
for key in keys:
    if refined_passages[key]["text"] in texts:
        print(f"Duplicate passages found for key: {key}. Removing entry.")
        del refined_passages[key]
    else:
        texts.add(refined_passages[key]["text"])
del texts

with open("refined_passages.json", "w") as f:
    json.dump(refined_passages, f, indent=4)


Duplicate passages found for key: countries|Australia|Prime Minister of Australia. Removing entry.
Duplicate passages found for key: countries|Pakistan|Prime Minister of Pakistan. Removing entry.
Duplicate passages found for key: countries|Iraq|Prime Minister of Iraq. Removing entry.


In [9]:
import requests
from urllib.parse import urljoin 
from bs4 import BeautifulSoup
from tqdm import tqdm
from PIL import Image
from io import BytesIO
refined_passages = {}

with open("refined_passages.json", "r") as f:
    refined_passages = json.load(f)

def get_wikipedia_infobox_image(page_url: str) -> str | None:
    """
    Returns the main image URL from the Wikipedia infobox (right-side box).
    If no infobox image is found, returns None.
    """
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(page_url, headers=headers)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    # Find the infobox table
    infobox = soup.find("table", class_="infobox")
    if not infobox:
        print(f"No infobox found for {page_url}")
        return None

    # Find the first image inside the infobox
    img = infobox.find("img")
    if not img or not img.get("src"):
        print(f"No image found in infobox for {page_url}")
        return None

    # Convert relative URL to absolute
    image_url = urljoin(page_url, img["src"])

    image_url_refined = re.sub(
        r"/thumb|(\.(svg|png|jpg|jpeg|tif)).*",
        r"\1",
        image_url,
        flags=re.IGNORECASE
    )
    return image_url_refined


for key in tqdm(refined_passages.keys(), desc="Fetching images from webpages"):
    page_url = refined_passages[key]["page_url"]
    image_url = get_wikipedia_infobox_image(page_url)
    refined_passages[key]["image_url"] = image_url

with open("refined_passages_with_images.json", "w") as f:
    json.dump(refined_passages, f, indent=4)

Fetching images from webpages:  94%|█████████▍| 49/52 [00:31<00:01,  2.52it/s]

No infobox found for https://en.wikipedia.org/wiki/Koji_Sato_(engineer)


Fetching images from webpages: 100%|██████████| 52/52 [00:33<00:00,  1.56it/s]


In [10]:

import json
with open("refined_passages_with_images.json", "r") as f:
    refined_passages = json.load(f)

refined_passages_to_save = {}

for key, value in refined_passages.items():
    category, element, *rest = key.split("|")
    key = None
    if category == "athletes":
        key = "|".join([category, element, "Sports Team", *rest])
    else:
        key = "|".join([category, element, *rest])
    refined_passages_to_save[key] = value

with open("final_refined_passages_with_images.json", "w") as f:
    json.dump(refined_passages_to_save, f, indent=4)